In [1]:
# ======================
# 摄像头视频
# 人像分割 + 换背景 + 漫画GAN
# Ascend ACL 稳定版
# ======================

import sys
import os
import time
import numpy as np
import cv2
import acl
import acllite_utils as utils
import constants as const

from acllite_imageproc import AclLiteImageProc
from acllite_model import AclLiteModel
from acllite_image import AclLiteImage
from acllite_resource import resource_list

# ======================
# ACL 资源初始化（昇腾环境）
# ======================
class AclLiteResource:
    def __init__(self, device_id=0):
        self.device_id = device_id
        self.context = None
        self.stream = None
        self.run_mode = None

    def init(self):
        acl.init()
        acl.rt.set_device(self.device_id)
        self.context, ret = acl.rt.create_context(self.device_id)
        self.stream, ret = acl.rt.create_stream()
        self.run_mode, ret = acl.rt.get_run_mode()

    def __del__(self):
        resource_list.destroy()
        if self.stream:
            acl.rt.destroy_stream(self.stream)
        if self.context:
            acl.rt.destroy_context(self.context)
        acl.rt.reset_device(self.device_id)

# ======================
# 人像分割模型
# ======================
class Seg(object):
    def __init__(self, model_path, model_width, model_height):
        self._model_path = model_path
        self._model_width = model_width
        self._model_height = model_height
        self._dvpp = None
        self._model = None

    def init(self):
        self._dvpp = AclLiteImageProc()
        self._model = AclLiteModel(self._model_path)
        return const.SUCCESS

    # 预处理：缩放 + 格式转换
    def pre_process(self, image):
        image_dvpp = image.copy_to_dvpp()
        yuv_image = self._dvpp.jpegd(image_dvpp)
        resized_image = self._dvpp.resize(yuv_image, self._model_width, self._model_height)
        return resized_image

    # 模型推理
    def inference(self, input_data):
        return self._model.execute(input_data)

    # 后处理：输出人像掩码
    def post_process(self, infer_output):
        data = infer_output[0]
        vals = data.flatten()
        mask = np.clip((vals * 255), 0, 255).reshape(224, 224, 2)
        return mask[:, :, 0]

# ======================
# 漫画GAN风格转换
# ======================
class CartoonGAN:
    def __init__(self, model_path):
        self.model = AclLiteModel(model_path)

    # 预处理：归一化 + 尺寸调整 + 通道转换
    def preprocess(self, frame):
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        self.h, self.w = img.shape[:2]
        img = cv2.resize(img, (256, 256))
        img = img.astype(np.float32) / 127.5 - 1.0
        img = np.transpose(img, (2, 0, 1))
        return np.expand_dims(img, axis=0)

    # 推理
    def inference(self, img):
        return self.model.execute([img])

    # 后处理：恢复图像尺寸 + 格式
    def postprocess(self, output):
        data = np.squeeze(output[0])
        data = np.transpose(data, (1, 2, 0))
        data = ((data + 1) * 127.5).astype(np.uint8)
        data = cv2.cvtColor(data, cv2.COLOR_RGB2BGR)
        return cv2.resize(data, (self.w, self.h))

    # 整体漫画化接口
    def cartoonize(self, frame):
        img = self.preprocess(frame)
        output = self.inference(img)
        return self.postprocess(output)

# ======================
# 人像 + 新背景 融合
# ======================
def merge_with_background(frame, mask, bg_img):
    h, w = frame.shape[:2]
    bg_resized = cv2.resize(bg_img, (w, h))
    mask = cv2.resize(mask, (w, h)) / 255.0
    mask = 1.0 - mask
    mask_3c = np.repeat(mask[:, :, np.newaxis], 3, axis=2)
    result = (bg_resized * (1 - mask_3c) + frame * mask_3c).astype(np.uint8)
    return result

# ======================
# 摄像头读取 + 视频保存 + AI处理
# ======================
def record_and_process_video(seg, cartoon, bg_path, save_raw_path, save_final_path, duration=8):
    bg_img = cv2.imread(bg_path)

    # 自动查找摄像头
    cap = None
    for idx in range(10):
        test_cap = cv2.VideoCapture(idx)
        if test_cap.isOpened():
            cap = test_cap
            break

    fps = 2
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    out_raw = cv2.VideoWriter(save_raw_path, fourcc, fps, (width, height))
    out_final = cv2.VideoWriter(save_final_path, fourcc, fps, (width, height))

    start_time = time.time()
    while time.time() - start_time < duration:
        ret, frame = cap.read()
        if not ret: break

        out_raw.write(frame)

        # 人像分割
        cv2.imwrite("/tmp/tmp_frame.jpg", frame)
        img = AclLiteImage("/tmp/tmp_frame.jpg")
        pre = seg.pre_process(img)
        out = seg.inference([pre])
        mask = seg.post_process(out)

        # 换背景
        merged_frame = merge_with_background(frame, mask, bg_img)

        # 漫画化
        cartoon_frame = cartoon.cartoonize(merged_frame)

        out_final.write(cartoon_frame)

    cap.release()
    out_raw.release()
    out_final.release()
    print("✅ 视频处理完成")

# ======================
# 主函数
# ======================
def main():
    # 自动创建输出文件夹 work/out/result
    os.makedirs(IMAGE_DIR, exist_ok=True)

    # 初始化ACL环境
    acl_res = AclLiteResource()
    acl_res.init()

    # 初始化人像分割
    seg = Seg(SEG_MODEL_PATH, MODEL_WIDTH, MODEL_HEIGHT)
    seg.init()

    # 初始化漫画GAN
    cartoon = CartoonGAN(CARTOON_MODEL_PATH)

    # 背景图仍放在原data目录，视频输出到out/result
    record_and_process_video(
        seg, cartoon,
        bg_path="./data/background.jpg",
        save_raw_path="./out/result/me_raw.mp4",
        save_final_path="./out/result/me_cartoon.mp4",
        duration=10
    )

# ======================
# 路径全局配置
# ======================
currentPath = '.'
# 模型路径不变
SEG_MODEL_PATH = os.path.join(currentPath, "./model/portrait.om")
CARTOON_MODEL_PATH = "./model/generatorv2.om"
# 输出文件统一存放路径
IMAGE_DIR = "./out/result"
# 模型输入尺寸
MODEL_WIDTH = 224
MODEL_HEIGHT = 224

# ======================
# 程序入口
# ======================
if __name__ == "__main__":
    main()

Init model resource start...
[AclLiteModel] create model output dataset:
malloc output 0, size 401408
Create model output dataset success
Init model resource success
Init model resource start...
[AclLiteModel] create model output dataset:
malloc output 0, size 393216
Create model output dataset success
Init model resource success
✅ 视频处理完成
dvpp resource release success
AclLiteModel release source success
AclLiteModel release source success
